# Ceros: métodos escalares

Cambia f, df, intervalo o semillas. Cada bloque es independiente. Residuo pequeño no garantiza error absoluto pequeño.

**Uso:** ejecuta los imports y el bloque del caso que necesites. Cada ejemplo empieza con sus propios datos. `def` se usa solo para expresiones matemáticas.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Bisección

Continuidad y cambio de signo. El semiancho del intervalo acota el error del punto medio.

In [2]:
def f(x):
    return x*np.sin(x)-1
a,b=0.,2.
tol=1e-8; max_iter=200
fa,fb=f(a),f(b)
assert a<b and (fa==0 or fb==0 or np.sign(fa)!=np.sign(fb))
convergio=False
if fa==0 or fb==0:
    raiz=a if fa==0 else b
    iteraciones=0; convergio=True
else:
    for i in range(max_iter):
        c=(a+b)/2
        fc=f(c); iteraciones=i+1
        if fc==0 or (b-a)/2<=tol:
            convergio=True
            break
        if np.sign(fa)!=np.sign(fc):
            b=c; fb=fc
        else:
            a=c; fa=fc
    raiz=c
print(raiz,iteraciones,'residuo:',abs(f(raiz)),'convergió:',convergio)
assert convergio

1.114157147705555 28 residuo: 9.490602304040863e-09 convergió: True


## 2. Posición falsa

Mismo requisito de signos; utiliza el corte de la secante con el eje x. Puede estancarse.

In [3]:
def f(x):
    return x*np.sin(x)-1
a,b=0.,2.
tol=1e-8; max_iter=200
fa,fb=f(a),f(b)
assert a<b and (fa==0 or fb==0 or np.sign(fa)!=np.sign(fb))
convergio=False
if fa==0 or fb==0:
    raiz=a if fa==0 else b
    iteraciones=0; convergio=True
else:
    for i in range(max_iter):
        c=a-fa*(b-a)/(fb-fa)
        fc=f(c); iteraciones=i+1
        if abs(fc)<=tol:
            convergio=True
            break
        if np.sign(fa)!=np.sign(fc):
            b=c; fb=fc
        else:
            a=c; fa=fc
    raiz=c
print(raiz,iteraciones,'residuo:',abs(f(raiz)),'convergió:',convergio)
assert convergio

1.1141571430336825 4 residuo: 3.0022619945668794e-09 convergió: True


## 3. Newton-Raphson

Una semilla y derivada; revisar derivada nula, divergencia, paso y residuo.

In [4]:
def f(x):
    return x*np.sin(x)-1
def df(x):
    return np.sin(x)+x*np.cos(x)
x=1.; tol=1e-8; max_iter=100
convergio=False
for i in range(max_iter):
    fx=f(x); dfx=df(x)
    if fx==0:
        convergio=True; break
    if dfx==0:
        raise ValueError('Derivada nula')
    x_nuevo=x-fx/dfx
    paso=abs(x_nuevo-x)
    x=x_nuevo
    if not np.isfinite(x):
        break
    if paso<=tol and abs(f(x))<=tol:
        convergio=True; break
print(x,'residuo:',abs(f(x)),'convergió:',convergio)
assert convergio

1.1141571408719302 residuo: 2.220446049250313e-16 convergió: True


## 4. Secante

Dos semillas. Actualiza ambas al terminar la iteración.

In [5]:
def f(x):
    return x*np.sin(x)-1
x0,x1=1.,1.5
tol=1e-8; max_iter=100
convergio=False
for i in range(max_iter):
    f0,f1=f(x0),f(x1)
    if f1==0:
        convergio=True; break
    if f1==f0:
        raise ValueError('Denominador nulo')
    x2=x1-f1*(x1-x0)/(f1-f0)
    paso=abs(x2-x1)
    x0,x1=x1,x2
    if not np.isfinite(x1):
        break
    if paso<=tol and abs(f(x1))<=tol:
        convergio=True; break
print(x1,'residuo:',abs(f(x1)),'convergió:',convergio)
assert convergio

1.1141571408719302 residuo: 2.220446049250313e-16 convergió: True


## 5. Varias raíces

Barrido y bisección. Puede omitir raíces dobles o muy juntas; una discontinuidad puede causar cambio de signo sin raíz.

In [6]:
def f(x):
    return (x-.23)*(x-.77)
a,b,N=0.,1.,80
x=np.linspace(a,b,N+1); y=f(x)
intervalos=[]; raices=list(x[y==0])
for i in range(N):
    if y[i]!=0 and y[i+1]!=0 and np.sign(y[i])!=np.sign(y[i+1]):
        intervalos.append((x[i],x[i+1]))
for a,b in intervalos:
    fa=f(a)
    for i in range(200):
        c=(a+b)/2; fc=f(c)
        if fc==0 or (b-a)/2<=1e-9:
            break
        if np.sign(fa)!=np.sign(fc):
            b=c
        else:
            a=c; fa=fc
    raices.append(c)
raices=np.sort(raices)
print('Intervalos:',intervalos,'Raíces:',raices)
assert np.allclose(raices,[.23,.77])

Intervalos: [(np.float64(0.225), np.float64(0.23750000000000002)), (np.float64(0.7625000000000001), np.float64(0.775))] Raíces: [0.23 0.77]


Material de estudio adaptado del notebook de clases y de la versión FC2_C2. Ejemplos de práctica; no son soluciones de las tareas. Comprueba las reglas de uso de material asistido por IA.